# Importing Libraries

In [ ]:
from torch_pdegraph.utilities import *
import numpy as np
from matplotlib import image as mpimg
import faiss
import matplotlib.pyplot as plt
import torch
import numpy as np
import torch
from skimage.util import view_as_windows
from matplotlib import image as mpimg

# Function to save the images

In [5]:
def saveImg(img, file_path):
    """
    Saves the image to the specified file path after normalizing it. 
    If the image is 3D with identical color channels, it converts it to grayscale before saving.
    
    param img: The input image, can be 2D or 3D array.
    param file_path: The path where to save the image, including the file name and extension.
    """
    # Check if img is a 2D grayscale or a 3D RGB where all channels are identical
    if len(img.shape) == 3 and np.all(img[:, :, 0] == img[:, :, 1]) and np.all(img[:, :, 1] == img[:, :, 2]):
        # If 3D RGB with identical channels, take the first channel to convert to grayscale
        img = img[:, :, 0]

    # Normalize the image to 0-1 range if it's not already in uint8 format
    if img.dtype != np.uint8:
        img = (img - np.min(img)) / (np.max(img) - np.min(img))

    # Save the image
    mpimg.imsave(file_path, img, cmap='gray')

# Load and Process the image

In [ ]:
image_path = "../data/barbara.png" # Input the correct image image path here
x = mpimg.imread(image_path)
if x.ndim == 3:
    x = np.dot(x[..., :3], [0.2989, 0.5870, 0.1140])  # Convert to grayscale
x = addNoise(7,x)
dispImg(x)

# Create the graph (tensor)

In [7]:
neighborhood_size = 21  # e.g., 15x15 neighborhood
patch_size = 7  # e.g., 5x5 patches
sigma = 0.3

# Padding the image to handle patches at the edges
pad_width = patch_size // 2
padded_image = np.pad(x, pad_width, mode='reflect')

# Extract patches
patches = view_as_windows(padded_image, (patch_size, patch_size))
# Flatten each patch into a vector
patches_flat = patches.reshape(-1, patch_size * patch_size)

# Image dimensions
rows, cols = x.shape

# Initialize lists for edges and weights
edge_list = []
weights = []

# Calculate half the neighborhood size for indexing
half_n_size = neighborhood_size // 2

# Iterate over each pixel in the image
for i in range(rows):
    for j in range(cols):
        center_patch_idx = (i * cols) + j
        center_patch = patches_flat[center_patch_idx]

        # Determine the neighborhood bounds
        for di in range(-half_n_size, half_n_size + 1):
            for dj in range(-half_n_size, half_n_size + 1):
                ni, nj = i + di, j + dj
                # Skip out-of-bounds and the center pixel itself
                if (0 <= ni < rows) and (0 <= nj < cols):
                    neighbor_patch_idx = (ni * cols) + nj
                    neighbor_patch = patches_flat[neighbor_patch_idx]

                    # Calculate L2 distance between the center patch and neighbor patch
                    dist = np.linalg.norm(center_patch - neighbor_patch)

                    # Compute weight based on L2 distance
                    weight = np.exp(-dist/(sigma*sigma))

                    # Append to the edge list and weight list
                    edge_list.append([center_patch_idx, neighbor_patch_idx])
                    weights.append(weight)

# Convert edge list and weights to PyTorch tensors
edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous().to('cuda:0')
edge_attr = torch.tensor(weights, dtype=torch.float).view(-1, 1).to('cuda:0')

# Assuming a Graph class is defined to take these as input
graph = Graph(edge_index, edge_attr)

# Save the graph (if needed)

In [8]:
torch.save({'edge_attr':edge_attr, 'edge_index': edge_index},'../tensors/biggerGraph/tensors_lena.pt')

# Load the graph (if needed)

In [4]:
tensors = torch.load('../tensors/biggerGraph/tensors_lena.pt')
edge_attr = tensors['edge_attr']
edge_index = tensors['edge_index']
graph = Graph(edge_index, edge_attr)

# Run the iterations and display the images

In [ ]:
import importlib
import pLaplacianDiffusion

# After making changes to the pLaplacianDiffusion.py file
importlib.reload(pLaplacianDiffusion)

#hyperparams
hp = dict(x0=torch.tensor(toFmat(x),dtype=torch.float32).cuda(), lamb=0.8, alpha=0.65, beta=0.35)

itr = 20
features = torch.tensor(toFmat(x),dtype=torch.float32).cuda()

pde = pLaplacianDiffusion.PDE(graph, **hp)
new_features = pde(features, itr)
y = new_features.cpu().numpy()
y = toImg(y,x.shape)
z = mpimg.imread("../data/lena.png") # input the correct image path here
dispImg(x)
dispImg(y)
dispImg(z)

# Save the output image

In [24]:
saveImg(y,"../output/lena6.png") # Input the correct output path and filename here